In [0]:
from pyspark.sql.functions import *
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier
)

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator
)

import pandas as pd
import logging

In [0]:
df = spark.table(
    "fiap_analytics.gold.rw_db_churn_ml_dataset"
)

display(df.limit(5))

In [0]:
feature_columns = [

    "total_sessions",
    "avg_bet_amount",
    "total_deposit_amount",
    "bonus_usage_rate",
    "avg_session_length",
    "avg_games_played",
    "days_since_last_session",
    "weekend_activity_ratio"

]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df_model = assembler.transform(df)

model_df = df_model.select(

    "user_id",

    "features",

    col("churn").alias("label")
)

display(model_df.limit(5))

In [0]:
train_df, test_df = model_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print(f"Registros treino: {train_df.count()}")
print(f"Registros teste: {test_df.count()}")

In [0]:
rf = RandomForestClassifier(

    featuresCol="features",

    labelCol="label",

    numTrees=100,

    seed=42
)

rf_model = rf.fit(train_df)

rf_predictions = rf_model.transform(test_df)

extract_probability = udf(
    lambda x: float(x[1]),
    DoubleType()
)

rf_predictions_clean = rf_predictions.withColumn(

    "churn_probability",

    extract_probability(col("probability"))
)

rf_predictions_clean = rf_predictions_clean.withColumn(
    "risk_level",
    when(col("churn_probability") >= 0.80, "Critical")
    .when(col("churn_probability") >= 0.60, "High")
    .when(col("churn_probability") >= 0.30, "Medium")
    .otherwise("Low")
)

final_predictions = rf_predictions_clean.select(
    "user_id",
    "label",
    "prediction",
    round(
        col("churn_probability"),
        4
    ).alias("churn_probability"),
    "risk_level"
)

final_predictions = final_predictions.orderBy(
    col("churn_probability").desc()
)

display(final_predictions)

In [0]:
evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

rf_auc = evaluator.evaluate(rf_predictions)

print(f"AUC Random Forest: {rf_auc}")

In [0]:
final_predictions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "fiap_analytics.gold.rw_db_churn_predictions_clean"
    )

print("Tabela final salva com sucesso.")

In [0]:
%sql
SELECT * FROM fiap_analytics.gold.rw_db_churn_predictions_clean